In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import NNConv, GCNConv, global_mean_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import random
import scipy.stats as stats
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

data_list = torch.load("gnn_dataset_20250409_225849.pt", weights_only=False)
if isinstance(data_list, Data):
    data_list = [data_list]

train_data, test_data = train_test_split(data_list, test_size=0.2, random_state=42)
train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1)

In [2]:
class GNNPredictor(nn.Module):
    def __init__(self, node_in, edge_in, hidden_dim, qubit_dim=127, output_dim=1):
        super(GNNPredictor, self).__init__()

        # Define neural network layers
        self.nn1 = nn.Sequential(
            nn.Linear(edge_in, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_in * hidden_dim)
        )
        self.conv1 = NNConv(node_in, hidden_dim, self.nn1, aggr='mean')
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin1 = nn.Linear(hidden_dim + qubit_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, data):
        # Extract data attributes
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        batch = data.batch if hasattr(data, 'batch') else torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        # Apply the first graph convolution layer
        x = F.relu(self.conv1(x, edge_index, edge_attr))
        # Apply the second graph convolution layer
        x = F.relu(self.conv2(x, edge_index))
        # Apply global mean pooling to the node features
        x = global_mean_pool(x, batch)

        # Handle the calculation of joint expectation
        # When calculating joint expectation, use joint_noisy
        joint_noisy = data.joint_noisy.view(x.size(0), -1)  # shape [batch_size, 1]
        x = torch.cat([x, joint_noisy], dim=1)
        
        # Output should be a single value (scalar)
        return self.lin2(F.relu(self.lin1(x))).squeeze(1)  # Output shape [batch_size] (joint expectation)

# Define device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Sample data for initializing the model (replace with actual train_data)
sample_data = train_data[0]

# Initialize the model for joint expectation
model = GNNPredictor(
    node_in=sample_data.x.shape[1],
    edge_in=sample_data.edge_attr.shape[1],
    hidden_dim=64,
    qubit_dim=1,  # Since we're only calculating joint expectation, we use 1
    output_dim=1  # Output dimension is 1 for joint expectation
).to(device)

# Define the optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()


In [3]:
# Training loop
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Forward pass
        pred = model(batch)  # shape: [batch_size] (joint expectation)

        # Use joint_ideal when calculating joint expectation (shape: [batch_size, 1])
        target = batch.joint_ideal.to(device).view(-1)  # shape: [batch_size] (scalar)

        # Compute loss
        loss = criterion(pred, target)

        # Backpropagation
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1:03d} | Avg Loss: {avg_loss:.6f}")


Epoch 001 | Avg Loss: 0.805204
Epoch 002 | Avg Loss: 0.111455
Epoch 003 | Avg Loss: 0.048326
Epoch 004 | Avg Loss: 0.027878
Epoch 005 | Avg Loss: 0.019894
Epoch 006 | Avg Loss: 0.014487
Epoch 007 | Avg Loss: 0.011134
Epoch 008 | Avg Loss: 0.008892
Epoch 009 | Avg Loss: 0.005113
Epoch 010 | Avg Loss: 0.003045
Epoch 011 | Avg Loss: 0.001614
Epoch 012 | Avg Loss: 0.001072
Epoch 013 | Avg Loss: 0.001275
Epoch 014 | Avg Loss: 0.001013
Epoch 015 | Avg Loss: 0.000987
Epoch 016 | Avg Loss: 0.002443
Epoch 017 | Avg Loss: 0.001060
Epoch 018 | Avg Loss: 0.001032
Epoch 019 | Avg Loss: 0.000860
Epoch 020 | Avg Loss: 0.000898
Epoch 021 | Avg Loss: 0.000885
Epoch 022 | Avg Loss: 0.000799
Epoch 023 | Avg Loss: 0.000810
Epoch 024 | Avg Loss: 0.000716
Epoch 025 | Avg Loss: 0.000776
Epoch 026 | Avg Loss: 0.000749
Epoch 027 | Avg Loss: 0.000712
Epoch 028 | Avg Loss: 0.000634
Epoch 029 | Avg Loss: 0.000722
Epoch 030 | Avg Loss: 0.000657
Epoch 031 | Avg Loss: 0.000640
Epoch 032 | Avg Loss: 0.000611
Epoch 03

In [4]:
model.eval()
all_preds, all_targets, all_noisy = [], [], []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)

        # Forward pass
        pred = model(batch)

        # Joint expectation
        target = batch.joint_ideal.to(device)  # joint_ideal for joint expectation
        all_noisy.append(batch.joint_noisy.view(-1).cpu().numpy())  # Noisy values

        all_preds.append(pred.view(-1).cpu().numpy())
        all_targets.append(target.view(-1).cpu().numpy())

# Concatenate all batches
all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)
all_noisy = np.concatenate(all_noisy, axis=0)

# 8. MAE (Mean Absolute Error) Calculation
mae_pred = mean_absolute_error(all_targets, all_preds)
mae_noisy = mean_absolute_error(all_targets, all_noisy)  # MAE of Ideal - Noisy

print(f"\nTest MAE (Ideal vs Predicted): {mae_pred:.6f}")
print(f"Test MAE (Ideal vs Noisy): {mae_noisy:.6f}")



Test MAE (Ideal vs Predicted): 0.013246
Test MAE (Ideal vs Noisy): 0.024320


In [5]:
# # --- Compute Noisy Test Inputs ---
# z_noisy_test = []
# for batch in test_loader:
#     z_noisy_test.append(batch.z_noisy[:10].unsqueeze(0).numpy())  # shape [1, 10]
# z_noisy_test = np.concatenate(z_noisy_test, axis=0)  # shape [n_samples, 10]

# # --- Compute Errors ---
# error_noisy = np.abs(z_noisy_test - all_targets)   # |noisy - ideal|
# error_pred = np.abs(all_preds - all_targets)       # |pred - ideal|

# # --- Plot 1: Per-Qubit Predictions ---
# # --- Flatten and Mask ---
# z_ideal_flat = all_targets.flatten()
# pred_flat = all_preds.flatten()
# z_noisy_flat = z_noisy_test.flatten()

# # Mask: keep only valid entries
# valid_mask = (z_ideal_flat != -2)

# # Apply mask
# ideal_valid = z_ideal_flat[valid_mask]
# pred_valid = pred_flat[valid_mask]
# noisy_valid = z_noisy_flat[valid_mask]

# # --- Recover per-qubit indexing ---
# # Assume original shape: [num_samples, num_qubits]
# num_qubits = z_noisy_test.shape[1]
# valid_indices = np.where(valid_mask.reshape(-1))[0]  # 1D indices

# # Map flat index to (sample_idx, qubit_idx)
# sample_idx = valid_indices // num_qubits
# qubit_idx = valid_indices % num_qubits

# # Stack data
# data = np.stack([sample_idx, qubit_idx, ideal_valid, pred_valid, noisy_valid], axis=1)
# df = pd.DataFrame(data, columns=["sample", "qubit", "ideal", "pred", "noisy"])
# df["sample"] = df["sample"].astype(int)
# df["qubit"] = df["qubit"].astype(int)

# # --- Pick first 10 qubit indices that appear in the masked data ---
# qubits_to_plot = sorted(df["qubit"].unique())[:10]

# # --- Plot: Qubit-wise Z Predictions vs Ideal ---
# fig, axs = plt.subplots(2, 5, figsize=(20, 6), sharex=True, sharey=True)
# axs = axs.flatten()

# for i, q in enumerate(qubits_to_plot):
#     df_q = df[df["qubit"] == q].sort_values("sample")
#     axs[i].plot(df_q["sample"], df_q["ideal"], label='Ideal', marker='o', markersize=3, linestyle='-')
#     axs[i].plot(df_q["sample"], df_q["pred"], label='Predicted', marker='x', markersize=3, linestyle='--')
#     axs[i].set_title(f"Qubit {q}")
#     axs[i].grid(True)
#     if i % 5 == 0:
#         axs[i].set_ylabel("Z Expectation")
#     axs[i].set_xlabel("Test Sample")

# fig.suptitle("Qubit-wise Z Predictions vs Ideal (Masked)", fontsize=16)
# fig.legend(["Ideal", "Predicted"], loc='upper center', ncol=2, frameon=False)
# plt.tight_layout(rect=[0, 0, 1, 0.95])
# plt.show()


# # --- Plot 2: Error Mean ± Std per Qubit ---
# qubit_indices = np.arange(10)
# mean_noisy_err = error_noisy.mean(axis=0)
# std_noisy_err = error_noisy.std(axis=0)

# mean_pred_err = error_pred.mean(axis=0)
# std_pred_err = error_pred.std(axis=0)

# bar_width = 0.35
# plt.figure(figsize=(10, 5))
# plt.bar(qubit_indices - bar_width/2, mean_noisy_err, width=bar_width, yerr=std_noisy_err, label='|Noisy - Ideal|', capsize=4)
# plt.bar(qubit_indices + bar_width/2, mean_pred_err, width=bar_width, yerr=std_pred_err, label='|Pred - Ideal|', capsize=4)

# plt.xlabel("Qubit Index")
# plt.ylabel("Absolute Error")
# plt.title("Per-Qubit Mean ± Std of Prediction and Noisy Errors")
# plt.xticks(qubit_indices)
# plt.legend()
# plt.grid(True)
# plt.tight_layout()
# plt.show()
